# Violence Detection System — CNN + LSTM
**Architecture:**
```
Video Frames → MobileNetV2 (CNN, frozen) → TimeDistributed → LSTM × 2 → Dense → Sigmoid
```
- **MobileNetV2** extracts spatial features (1280-dim vector) from each frame  
- **LSTM × 2** learns temporal patterns across the 20-frame sequence  
- **Sigmoid output** → probability of violence (0 = calm, 1 = violent)  

**Dataset layout expected:**
```
data/
  violent/      ← .mp4 / .avi / .mov / .mkv clips
  nonviolent/
```

## 1. Install Requirements & Imports

In [ ]:
# Install dependencies (run once)
# !pip install tensorflow opencv-python numpy scikit-learn matplotlib

In [ ]:
import os
import sys
import numpy as np
import cv2
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.19.0


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohamedmustafa/real-life-violence-situations-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'real-life-violence-situations-dataset' dataset.
Path to dataset files: /kaggle/input/real-life-violence-situations-dataset


## 1. Configuration
All hyperparameters and paths are defined here. Edit this cell to customize training.

In [ ]:
CONFIG = {
    "sequence_len":    20,
    "img_size":        (224, 224),
    "batch_size":      32,           # Fast training ke liye batch bada kar sakte hain
    "epochs":          30,
    "lstm_units":      256,
    "dropout":         0.4,
    "learning_rate":   1e-4,
    "data_dir":        "/content/drive/MyDrive/extracted_features", # ✅ Drive wala path
    "model_save_path": "/content/drive/MyDrive/fast_violence_detector.keras",
}

# 2. Data Preprocessing

In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
import cv2

# 1. Tumhara purana frame extraction function
def extract_frames(video_path, seq_len=20, img_size=(224, 224)):
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total == 0:
        cap.release()
        return None
    indices = np.linspace(0, total - 1, seq_len, dtype=int)
    frames = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if not ret:
            frame = frames[-1] if frames else np.zeros((*img_size, 3), dtype=np.uint8)
        else:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame = cv2.resize(frame, img_size)
        frames.append(frame)
    cap.release()
    frames = np.array(frames, dtype=np.float32)
    return preprocess_input(frames)

# 2. Juicer (MobileNetV2) load karo
print("Loading MobileNetV2 Feature Extractor...")
cnn_extractor = MobileNetV2(weights="imagenet", include_top=False, pooling="avg")

# 3. Paths setup karo
# Yahan apni downloaded kaggle dataset ka path dalo
RAW_DATA_DIR = "/kaggle/input/real-life-violence-situations-dataset/Real Life Violence Dataset"
# Yeh naya folder hai jahan .npy files save hongi
FEATURES_DIR = "/kaggle/working/extracted_features"

for category in ["Violence", "NonViolence"]:
    os.makedirs(os.path.join(FEATURES_DIR, category), exist_ok=True)

    raw_folder = os.path.join(RAW_DATA_DIR, category)
    video_files = [f for f in os.listdir(raw_folder) if f.endswith(('.mp4', '.avi'))]

    print(f"Extracting features for {category}...")
    for video_name in video_files:
        video_path = os.path.join(raw_folder, video_name)
        save_path = os.path.join(FEATURES_DIR, category, video_name.split('.')[0] + ".npy")

        # Agar pehle se extracted hai toh skip karo
        if os.path.exists(save_path): continue

        frames = extract_frames(video_path)
        if frames is not None:
            # CNN se features nikalo aur save karo
            # ✅ SAHI LINE: (Seedha 20 frames ka batch pass karo)
            features = cnn_extractor.predict(frames, verbose=0)
            np.save(save_path, features)

print("Saari videos ke features .npy format mein save ho gaye!")

Loading MobileNetV2 Feature Extractor...


/tmp/ipykernel_13892/1168553187.py:32: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  cnn_extractor = MobileNetV2(weights="imagenet", include_top=False, pooling="avg")


Extracting features for Violence...
Extracting features for NonViolence...
Saari videos ke features .npy format mein save ho gaye!


In [ ]:
from google.colab import drive
import shutil

# 1. Google Drive connect karo (agar pehle se nahi kiya hai)
drive.mount('/content/drive')

print("Copying files to Google Drive... Please wait 1-2 minutes.")

# 2. Temporary folder se files uthao aur seedha Drive mein copy kar do
shutil.copytree('/kaggle/working/extracted_features', '/content/drive/MyDrive/extracted_features', dirs_exist_ok=True)

print("✅ Backup Complete! Ab tumhari files Google Drive mein hamesha ke liye safe hain.")

Mounted at /content/drive
Copying files to Google Drive... Please wait 1-2 minutes.
✅ Backup Complete! Ab tumhari files Google Drive mein hamesha ke liye safe hain.


## 3. Dataset Loader
Walks `data/violent/` and `data/nonviolent/`, extracts frame sequences from every video, and returns arrays `X` and `y`.

In [ ]:
class FeatureDataGenerator(tf.keras.utils.Sequence):
    def __init__(self, feature_paths, labels, batch_size, shuffle=True):
        self.feature_paths = feature_paths
        self.labels = labels
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.indices = np.arange(len(self.feature_paths))
        self.on_epoch_end()

    def __len__(self):
        return int(np.floor(len(self.feature_paths) / self.batch_size))

    def __getitem__(self, index):
        batch_indices = self.indices[index*self.batch_size:(index+1)*self.batch_size]
        X, y = [], []
        for i in batch_indices:
            # Seedha .npy file load kar rahe hain (Super Fast!)
            features = np.load(self.feature_paths[i])
            X.append(features)
            y.append(self.labels[i])

        return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

## 4. Model Architecture

| Layer | Output Shape | Notes |
|---|---|---|
| Input | `(batch, 20, 224, 224, 3)` | 20-frame sequence |
| TimeDistributed(MobileNetV2) | `(batch, 20, 1280)` | Same CNN on every frame |
| TimeDistributed(BatchNorm) | `(batch, 20, 1280)` | Stabilizes features |
| LSTM 1 | `(batch, 20, 256)` | `return_sequences=True` |
| LSTM 2 | `(batch, 128)` | Condenses to one vector |
| Dense(128, relu) | `(batch, 128)` | |
| Dropout | `(batch, 128)` | |
| Dense(1, sigmoid) | `(batch, 1)` | P(violent) |

In [ ]:
def build_fast_lstm_model(seq_len: int, feature_dim: int, lstm_units: int, dropout: float) -> tf.keras.Model:
    # Ab input image nahi hai, sirf 20 frames aur unke 1280 features hain
    inp = layers.Input(
        shape=(seq_len, feature_dim), # Yeh (20, 1280) hoga
        name="precomputed_features"
    )

    # Thoda data stabilize karne ke liye Batch Norm
    x = layers.BatchNormalization(name="bn_features")(inp)

    # Seedha LSTM layers! Koi CNN nahi.
    x = layers.LSTM(lstm_units,
                    return_sequences=True,
                    dropout=dropout,
                    name="lstm_1")(x)

    x = layers.LSTM(lstm_units // 2,
                    return_sequences=False,
                    dropout=dropout,
                    name="lstm_2")(x)

    x = layers.Dense(128, activation="relu", name="dense_1")(x)
    x = layers.Dropout(dropout, name="dropout_head")(x)
    out = layers.Dense(1, activation="sigmoid", name="output")(x)

    return models.Model(inputs=inp, outputs=out, name="LightningFast_ViolenceDetector")

# Preview the new architecture
model = build_fast_lstm_model(
    seq_len=CONFIG["sequence_len"],
    feature_dim=1280, # MobileNetV2 hamesha 1280 output deta hai
    lstm_units=CONFIG["lstm_units"],
    dropout=CONFIG["dropout"]
)
model.summary()

Model: "LightningFast_ViolenceDetector"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ precomputed_features            │ (None, 20, 1280)       │             0 │
│ (InputLayer)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn_features                     │ (None, 20, 1280)       │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 20, 256)        │     1,573,888 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 128)            │       197,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_head (Dropout)          │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,792,769 (6.84 MB)

 Trainable params: 1,790,209 (6.83 MB)

 Non-trainable params: 2,560 (10.00 KB)

## 5. Training Callbacks

| Callback | Purpose |
|---|---|
| `EarlyStopping` | Stops training when `val_auc` stops improving (patience = 6) |
| `ReduceLROnPlateau` | Halves LR when `val_loss` plateaus (patience = 3) |
| `ModelCheckpoint` | Saves the best model weights based on `val_auc` |

In [ ]:
def get_callbacks(model_save_path: str) -> list:
    return [
        callbacks.EarlyStopping(
            monitor="val_auc", patience=6,
            restore_best_weights=True, mode="max"
        ),
        callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6
        ),
        callbacks.ModelCheckpoint(
            filepath=model_save_path,
            monitor="val_auc", save_best_only=True, mode="max"
        ),
    ]

# 6. Main Training Engine

In [ ]:
def train_fast_model(config):
    # 1. Yahan videos ki jagah apni .npy files ke paths load karo
    feature_paths, labels = [], []
    class_map = {"Violence": 1, "NonViolence": 0}

    # DHYAAN DEIN: "data_dir" ab us folder ko point karna chahiye jahan tumne .npy save kiye hain
    for class_name, label in class_map.items():
        class_dir = os.path.join(config["data_dir"], class_name)
        # Ab hum .mp4 nahi, .npy dhoondh rahe hain
        files = [os.path.join(class_dir, f) for f in os.listdir(class_dir) if f.endswith('.npy')]
        feature_paths.extend(files)
        labels.extend([label] * len(files))

    # 2. Split paths
    train_paths, val_paths, train_labels, val_labels = train_test_split(
        feature_paths, labels, test_size=0.2, stratify=labels, random_state=42
    )

    # 3. Create Generators (Naya wala use kar rahe hain)
    train_gen = FeatureDataGenerator(train_paths, train_labels, config['batch_size'])
    val_gen = FeatureDataGenerator(val_paths, val_labels, config['batch_size'], shuffle=False)

    # 4. Build & Compile Model
    model = build_fast_lstm_model(config['sequence_len'], 1280, config['lstm_units'], config['dropout'])
    compile_model(model, config['learning_rate'])

    print("\n[Starting Super Fast LSTM Training]")
    # Ab yeh training rocket ki speed se hogi!
    model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=config['epochs'],
        callbacks=get_callbacks(config['model_save_path'])
    )

    return model

In [ ]:
trained_lstm_model = train_fast_model(CONFIG)


[Starting Super Fast LSTM Training]


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/30
50/50 ━━━━━━━━━━━━━━━━━━━━ 12s 176ms/step - accuracy: 0.6737 - auc: 0.7522 - loss: 0.6299 - val_accuracy: 0.8229 - val_auc: 0.9628 - val_loss: 0.5024 - learning_rate: 1.0000e-04
Epoch 2/30
50/50 ━━━━━━━━━━━━━━━━━━━━ 7s 150ms/step - accuracy: 0.8694 - auc: 0.9365 - loss: 0.4439 - val_accuracy: 0.8750 - val_auc: 0.9766 - val_loss: 0.3136 - learning_rate: 1.0000e-04
Epoch 3/30
50/50 ━━━━━━━━━━━━━━━━━━━━ 10s 192ms/step - accuracy: 0.9038 - auc: 0.9640 - loss: 0.2815 - val_accuracy: 0.9010 - val_auc: 0.9861 - val_loss: 0.2213 - learning_rate: 1.0000e-04
Epoch 4/30
50/50 ━━━━━━━━━━━━━━━━━━━━ 8s 151ms/step - accuracy: 0.9456 - auc: 0.9866 - loss: 0.1715 - val_accuracy: 0.9401 - val_auc: 0.9924 - val_loss: 0.1397 - learning_rate: 1.0000e-04
Epoch 5/30
50/50 ━━━━━━━━━━━━━━━━━━━━ 8s 163ms/step - accuracy: 0.9488 - auc: 0.9901 - loss: 0.1338 - val_accuracy: 0.9583 - val_auc: 0.9943 - val_loss: 0.1147 - learning_rate: 1.0000e-04
Epoch 6/30
50/50 ━━━━━━━━━━━━━━━━━━━━ 8s 155ms/step - accu

# Testing / Inference

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Model ko manual save karne ke liye
manual_save_path = "/content/drive/MyDrive/final_violence_model_v1.keras"
trained_lstm_model.save(manual_save_path)

print(f"✅ Model successfully saved at: {manual_save_path}")

✅ Model successfully saved at: /content/drive/MyDrive/final_violence_model_v1.keras


In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2

# 1. Apna trained LSTM model load karo
# Agar trained_lstm_model abhi memory mein hai toh ye line skip kar sakte ho
model = tf.keras.models.load_model("/content/drive/MyDrive/final_violence_model_v1.keras")

# 2. Features nikalne ke liye MobileNetV2 load karo (Inference ke liye zaroori hai)
feature_extractor = MobileNetV2(weights="imagenet", include_top=False, pooling="avg")

print("✅ Model aur Feature Extractor ready hain!")

/tmp/ipykernel_521/157882767.py:9: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  feature_extractor = MobileNetV2(weights="imagenet", include_top=False, pooling="avg")


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
✅ Model aur Feature Extractor ready hain!


In [ ]:
import cv2
import numpy as np
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

def predict_video(video_path, lstm_model, cnn_extractor, config, threshold=0.5):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"❌ Error: Video file nahi mili: {video_path}")
        return

    seq_len = config["sequence_len"]
    img_size = config["img_size"]
    buffer = []

    print(f"🎬 Analyzing Video: {video_path}")
    print("Please wait, results aa rahe hain...\n")

    while True:
        ret, frame = cap.read()
        if not ret: break

        # Frame preprocess karo (Wahi jo training mein kiya tha)
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        rgb = cv2.resize(rgb, img_size).astype(np.float32)
        rgb = preprocess_input(rgb)
        buffer.append(rgb)

        # Jab 20 frames ikhatte ho jayein
        if len(buffer) == seq_len:
            frames_array = np.array(buffer)

            # 1. CNN (Juicer) se features nikalo
            features = cnn_extractor.predict(frames_array, verbose=0)

            # 2. LSTM (Brain) se prediction lo
            features_batch = np.expand_dims(features, axis=0)
            score = float(lstm_model.predict(features_batch, verbose=0)[0][0])

            # 3. Result dikhao
            label = "⚠️ VIOLENCE DETECTED" if score >= threshold else "✅ NORMAL"
            print(f"Result: {label} | Confidence: {score:.4f}")

            # Slide window by half (10 frames) taaki processing fast ho
            buffer = buffer[seq_len // 2:]

    cap.release()
    print("\n✅ Analysis Complete.")

In [ ]:
# Mere bataye hue predict_video function ko use karte hue:

# 1. Video ka sahi path dalo (Google Drive ya local)
video_to_test = "/content/drive/MyDrive/hdfight.mp4"

# 2. Model, Extractor aur Config ke saath prediction chalao
predict_video(video_to_test, model, feature_extractor, CONFIG)

🎬 Analyzing Video: /content/drive/MyDrive/hdfight.mp4
Please wait, results aa rahe hain...

Result: ⚠️ VIOLENCE DETECTED | Confidence: 0.9739
Result: ⚠️ VIOLENCE DETECTED | Confidence: 0.9995
Result: ⚠️ VIOLENCE DETECTED | Confidence: 0.9996
Result: ⚠️ VIOLENCE DETECTED | Confidence: 0.9996
Result: ⚠️ VIOLENCE DETECTED | Confidence: 0.9998
Result: ⚠️ VIOLENCE DETECTED | Confidence: 0.9998
Result: ⚠️ VIOLENCE DETECTED | Confidence: 0.9997
Result: ⚠️ VIOLENCE DETECTED | Confidence: 0.9997
Result: ⚠️ VIOLENCE DETECTED | Confidence: 0.9997
Result: ⚠️ VIOLENCE DETECTED | Confidence: 0.9997
Result: ⚠️ VIOLENCE DETECTED | Confidence: 0.9996
Result: ⚠️ VIOLENCE DETECTED | Confidence: 0.9996
Result: ⚠️ VIOLENCE DETECTED | Confidence: 0.9950
Result: ⚠️ VIOLENCE DETECTED | Confidence: 0.9869
Result: ⚠️ VIOLENCE DETECTED | Confidence: 0.9993
Result: ⚠️ VIOLENCE DETECTED | Confidence: 0.9996
Result: ⚠️ VIOLENCE DETECTED | Confidence: 0.9997
Result: ⚠️ VIOLENCE DETECTED | Confidence: 0.9997
Result: 